# Reusable Forecasting Pipeline

Every step is a single function in the `src/` package, imported here rather
than redefined. An earlier version of this notebook kept its own copies of
`preprocess`, `split_data`, `evaluate_forecast` and `train_and_evaluate`;
those copies drifted from the ones in `02_modeling.ipynb`, which is how a
duplicated `weekly_mean_2` (identical to `rolling_mean_14`) and a missing
bank holiday went unnoticed.

The notebook runs the pipeline on one category, quantifies how much of the
result is noise, then applies it to all 54 categories.

In [2]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import time
import numpy as np
import pandas as pd

from src import config, features as feat, splits as spl, models as mdl
from src import pipeline as pipe, validation as val
from src.metrics import evaluate_forecast

daily = pd.read_csv(config.DAILY_CLEAN, parse_dates=["Date"])
daily = daily[~daily["Category"].isin(config.EXCLUDED_CATEGORIES)]

print(daily.shape, "|", daily["Category"].nunique(), "categories")
print(daily["Date"].min().date(), "->", daily["Date"].max().date())

(20196, 5) | 54 categories
2010-12-01 -> 2011-12-09


## 1. The pipeline, step by step

Four functions, each doing one thing.

`build_features` creates the target — units sold over X days, starting Y days
ahead — plus lags, rolling statistics, calendar and holiday features. Rolling
windows are shifted by one day so today's quantity never enters a feature
describing today.

`stratified_block_split` orders 14-day blocks by mean demand before dealing
them out, so each split receives a comparable share of high- and low-season
weeks. This matters more than it sounds: with plain shuffling, validation
WAPE on Christmas Decorations ranged from 17% to 209% depending only on the
random seed.

`leakage_ratio` measures the residual leakage the block split accepts. Two
rows leak into each other when their target windows overlap, i.e. when they
are fewer than X days apart.

In [4]:
CATEGORY = "Tealight Holders & Sets"
X, Y = 7, 7

data = feat.build_features(daily, CATEGORY, X=X, Y=Y)
target = config.target_name(X, Y)
features = feat.available_features(data)

print(f"{len(data)} usable rows, {len(features)} features")
print(f"target: {target}")

splits = spl.stratified_block_split(data, target, random_state=42)
print({k: len(v) for k, v in splits.items()})
print(f"leakage ratio: {spl.leakage_ratio(splits, X):.1%}")

333 usable rows, 30 features
target: target_X7_Y7
{'train': 238, 'validation': 53, 'test': 42}
leakage ratio: 57.1%


## 2. All models on one split

`train_all_models` runs every model in the registry. Adding a model means
writing one function with a `(train, val, test, features, target)` signature
and decorating it — nothing in this notebook changes.

TabFM is registered on demand and is evaluated separately in
`04_tabfm.ipynb`, since it requires a GPU to be practical.

In [6]:
print("registered models:", mdl.model_names())

results, predictions = pipe.train_all_models(splits, features, target)
results[["Model", "Validation WAPE (%)", "Validation R2",
         "Test WAPE (%)", "Test R2"]].sort_values("Validation WAPE (%)")

registered models: ['7-Day Rolling Sum Baseline', 'Linear Regression', 'Random Forest', 'XGBoost']


,Model,Validation WAPE (%),Validation R2,Test WAPE (%),Test R2
1,Linear Regression,15.389,0.757,31.024,-0.178
2,Random Forest,15.596,0.689,22.657,0.300
0,7-Day Rolling Sum Baseline,15.813,0.702,42.482,-1.171
3,XGBoost,17.032,0.610,24.904,0.218


**Read that table carefully.** The model ranked first on validation is not
the one ranked first on test. That is not a coincidence of this category, and
sections 3 and 4 quantify how often it happens.

## 3. How much of this is noise?

The table above comes from one split. With 374 days there are roughly 24
blocks, of which validation receives about four — too few to separate models
whose WAPE differs by a few points.

`repeated_evaluation` re-draws the split across 30 seeds and evaluates every
model on each draw. `win_rate` is the fraction of draws a model came first:
a model winning 40% of draws is not "the best model", it is the one that
happened to win a single draw.

In [9]:
rep = val.repeated_evaluation(daily, CATEGORY, X=X, Y=Y,
                              seeds=range(30), stratified=True)
val.summarise_repeats(rep)

,Category,Model,wape_mean,wape_std,wape_min,wape_max,r2_mean,win_rate
2,Tealight Holders & Sets,Random Forest,17.512,3.566,13.369,27.486,0.659,0.467
3,Tealight Holders & Sets,XGBoost,18.111,4.231,12.920,30.525,0.607,0.367
0,Tealight Holders & Sets,7-Day Rolling Sum Baseline,22.988,6.171,14.504,34.522,0.319,0.100
1,Tealight Holders & Sets,Linear Regression,25.544,5.908,15.365,37.667,0.268,0.067


`is_difference_meaningful` compares two models on the **same** 30 splits.
The pairing matters: both models see identical data on each seed, so the
per-seed difference cancels the split-to-split variation that dominates the
absolute numbers.

A negative `mean_diff` favours the first model. If the confidence interval
crosses zero, the two are indistinguishable on this dataset and no claim
should be made either way.

In [11]:
pairs = [("Random Forest", "7-Day Rolling Sum Baseline"),
         ("XGBoost", "Random Forest"),
         ("Linear Regression", "Random Forest")]

for a, b in pairs:
    d = val.is_difference_meaningful(rep, a, b)
    decided = (d["ci_low"] < 0) == (d["ci_high"] < 0)
    verdict = ("first better" if decided and d["mean_diff"] < 0
               else "second better" if decided else "within noise")
    print(f"{a:20s} vs {b:28s} {d['mean_diff']:+7.2f} "
          f"CI[{d['ci_low']:+6.2f},{d['ci_high']:+6.2f}]  {verdict}")

Random Forest        vs 7-Day Rolling Sum Baseline     -5.48 CI[ -7.79, -3.16]  first better
XGBoost              vs Random Forest                  +0.60 CI[ +0.00, +1.20]  second better
Linear Regression    vs Random Forest                  +8.03 CI[ +6.13, +9.94]  second better


## 4. All 54 categories

`run_pipeline` wraps everything above. Running it across every category turns
"is model selection reliable?" into a measurable quantity.

X and Y are held at 7 deliberately. WAPE is not comparable across different
values of X — the denominator grows with the window — so optimising horizons
per category would make the medians below meaningless.

In [13]:
rows, t0 = [], time.time()

for cat in sorted(daily["Category"].unique()):
    res = pipe.run_pipeline(daily, cat, X=7, Y=7, verbose=False, stratified=True)
    if res is None:
        continue
    r = res["results"]
    best_val = r.loc[r["Validation WAPE (%)"].idxmin()]
    best_test = r.loc[r["Test WAPE (%)"].idxmin()]
    rows.append({
        "Category": cat,
        "selected_model": best_val["Model"],
        "val_WAPE": best_val["Validation WAPE (%)"],
        "test_WAPE": best_val["Test WAPE (%)"],
        "test_R2": best_val["Test R2"],
        "best_on_test": best_test["Model"],
        "best_test_WAPE": best_test["Test WAPE (%)"],
        "selection_loss": round(best_val["Test WAPE (%)"]
                                - best_test["Test WAPE (%)"], 2),
        "selection_correct": best_val["Model"] == best_test["Model"],
    })

allcat = pd.DataFrame(rows)
allcat.to_csv(config.RESULTS / "pipeline_results_all_categories.csv", index=False)
print(f"{len(allcat)} categories in {time.time()-t0:.0f}s")

54 categories in 139s


In [14]:
print(f"selection picked the test winner: "
      f"{allcat['selection_correct'].sum()}/{len(allcat)} "
      f"({allcat['selection_correct'].mean()*100:.0f}%)")
print(f"random choice among 4 models would give ~25%")
print()
print(f"median test WAPE: {allcat['test_WAPE'].median():.1f}%")
print(f"positive test R2: {(allcat['test_R2']>0).sum()}/{len(allcat)}")
print()
comparison = pd.DataFrame({
    "selected on validation": allcat["selected_model"].value_counts(),
    "actual test winner": allcat["best_on_test"].value_counts(),
})
comparison

selection picked the test winner: 13/54 (24%)
random choice among 4 models would give ~25%

median test WAPE: 36.5%
positive test R2: 20/54



,selected on validation,actual test winner
7-Day Rolling Sum Baseline,10,11
Linear Regression,10,11
Random Forest,18,21
XGBoost,16,11


**Selection is no better than chance.** And the distribution of what it
costs is strongly right-skewed — usually harmless, occasionally severe.

In [16]:
loss = allcat["selection_loss"]
print(f"selection loss (test WAPE points lost by choosing on validation)")
print(f"  median   {loss.median():6.2f}")
print(f"  mean     {loss.mean():6.2f}")
print(f"  90th pct {loss.quantile(0.90):6.2f}")
print(f"  max      {loss.max():6.2f}")
print()
print(f"under 2 points: {(loss < 2).sum()}/{len(allcat)}")
print(f"over 10 points: {(loss > 10).sum()}/{len(allcat)}")
print()
allcat.nlargest(5, "selection_loss")[
    ["Category", "selected_model", "test_WAPE",
     "best_on_test", "best_test_WAPE", "selection_loss"]
]

selection loss (test WAPE points lost by choosing on validation)
  median     6.13
  mean      10.32
  90th pct  33.47
  max       50.00

under 2 points: 18/54
over 10 points: 18/54



,Category,selected_model,test_WAPE,best_on_test,best_test_WAPE,selection_loss
16,Easter Decorations,Linear Regression,73.627,7-Day Rolling Sum Baseline,23.631,50.00
29,"Jigsaw Puzzles, Card Games & Board Games",Linear Regression,68.976,XGBoost,28.984,39.99
17,Egg Cups & Holders,7-Day Rolling Sum Baseline,84.842,XGBoost,45.069,39.77
23,Hair Accessories,7-Day Rolling Sum Baseline,87.998,Random Forest,51.256,36.74
38,Mirrors,XGBoost,70.899,Linear Regression,37.004,33.90


## 5. Is a fixed model better than selecting?

If per-category selection is unreliable, the alternative is to apply one
model everywhere. The oracle column is the unreachable lower bound — the best
model on test, known only after the fact — and it sets the scale of what
perfect selection could possibly gain.

In [18]:
recs = []
for cat in sorted(daily["Category"].unique()):
    data_c = feat.build_features(daily, cat, X=7, Y=7)
    if len(data_c) < 50:
        continue
    tgt = config.target_name(7, 7)
    feats = feat.available_features(data_c)
    s = spl.stratified_block_split(data_c, tgt, random_state=42)
    rec = {"Category": cat}
    for name in mdl.model_names():
        _, vp, tp = mdl.MODELS[name](s["train"], s["validation"], s["test"],
                                     feats, tgt)
        rec[f"val_{name}"] = evaluate_forecast(s["validation"][tgt], vp)["wape"]
        rec[f"test_{name}"] = evaluate_forecast(s["test"][tgt], tp)["wape"]
    recs.append(rec)

strat = pd.DataFrame(recs)
strat.to_csv(config.RESULTS / "strategy_comparison.csv", index=False)

names = mdl.model_names()
test_cols = [f"test_{n}" for n in names]
val_cols = [f"val_{n}" for n in names]

oracle = strat[test_cols].min(axis=1)
chosen = strat.apply(
    lambda r: r[f"test_{names[[r[c] for c in val_cols].index(min(r[c] for c in val_cols))]}"],
    axis=1)

summary = [{"strategy": "select per category (validation)",
            "median": round(chosen.median(), 1), "mean": round(chosen.mean(), 1),
            ">10 pts off oracle": int(((chosen - oracle) > 10).sum())}]
for n in names:
    col = strat[f"test_{n}"]
    summary.append({"strategy": f"always {n}",
                    "median": round(col.median(), 1), "mean": round(col.mean(), 1),
                    ">10 pts off oracle": int(((col - oracle) > 10).sum())})
summary.append({"strategy": "oracle (best on test)",
                "median": round(oracle.median(), 1), "mean": round(oracle.mean(), 1),
                ">10 pts off oracle": 0})

pd.DataFrame(summary).set_index("strategy")

,median,mean,>10 pts off oracle
strategy,,,
select per category (validation),36.5,42.8,18
always 7-Day Rolling Sum Baseline,43.2,49.3,32
always Linear Regression,44.8,51.2,28
always Random Forest,35.8,38.0,10
always XGBoost,36.0,39.0,13
oracle (best on test),30.1,32.5,0


## Conclusion

Applying a single model to every category beats selecting one per category,
on both average WAPE and the number of catastrophic errors. Per-category
selection does not merely fail to help — it degrades results, adding variance
without adding information.

The oracle sits only a few points below the best fixed model, which explains
why: the prize for perfect selection is small relative to the error in
estimating it. Selection becomes worthwhile only with enough data to estimate
it reliably, and one year is not enough.

Operationally this also means one model to train, deploy and monitor instead
of 54.

Full write-up in `RESULTS_54_AND_TABFM.md`.